In [1]:
!pip install -q torch torchvision scikit-learn bloqade

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.7/178.7 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.3/21.3 MB 51.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.0/13.0 MB 67.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 313.7/313.7 kB 27.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.2/60.2 kB 6.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 917.8/917.8 kB 50.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.9/137.9 kB 9.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.4/41.4 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.6/125.6 kB 13.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.0/224.0 kB 21.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 191.7/191.7 kB 17.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.2/139.2 kB 12.8 MB/s eta 0:00:00
     ━━━━━━━━━━━

In [2]:
import numpy as np
# import juliacall
# bloqade_jl = juliacall.Main.seval('Bloqade')
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from sklearn.decomposition import PCA
from sklearn.preprocessing import OneHotEncoder
import bloqade
from bloqade.atom_arrangement import Chain
from bloqade.emulate.ir.space import Space
from bloqade.ir.routine.bloqade import BloqadeEmulation
from bloqade.factory import rydberg_h
# from bloqade import evolve

In [3]:
from bloqade import (
    waveform,
    rydberg_h,
    piecewise_linear,
    piecewise_constant,
    constant,
    linear,
    var,
    cast,
    start,
    get_capabilities,
)
from bloqade.atom_arrangement import Chain
from bloqade.ir import (
    AnalogCircuit,
    Sequence,
    rydberg,
    Pulse,
    rabi,
    detuning,
    Field,
    Uniform,
)
from bloqade.ir.routine.base import Routine
from bloqade.ir.routine.params import Params, ScalarArg

In [4]:

# Define constants
dim_pca = 10
Δ_max = 6.0
num_examples = 1000
num_test_examples = 100

In [5]:
# Load MNIST dataset
transform = transforms.Compose([transforms.ToTensor()])
train_dataset = datasets.MNIST('~/.pytorch/MNIST_data/', download=True, train=True, transform=transform)
test_dataset = datasets.MNIST('~/.pytorch/MNIST_data/', download=True, train=False, transform=transform)

Failed to download (trying next):
HTTP Error 403: Forbidden



100%|██████████| 9912422/9912422 [00:00<00:00, 16350861.36it/s]


Extracting /root/.pytorch/MNIST_data/MNIST/raw/train-images-idx3-ubyte.gz to /root/.pytorch/MNIST_data/MNIST/raw

Failed to download (trying next):
HTTP Error 403: Forbidden



100%|██████████| 28881/28881 [00:00<00:00, 478986.53it/s]


Extracting /root/.pytorch/MNIST_data/MNIST/raw/train-labels-idx1-ubyte.gz to /root/.pytorch/MNIST_data/MNIST/raw

Failed to download (trying next):
HTTP Error 403: Forbidden



100%|██████████| 1648877/1648877 [00:00<00:00, 4345689.71it/s]


Extracting /root/.pytorch/MNIST_data/MNIST/raw/t10k-images-idx3-ubyte.gz to /root/.pytorch/MNIST_data/MNIST/raw

Failed to download (trying next):
HTTP Error 403: Forbidden



100%|██████████| 4542/4542 [00:00<00:00, 5953290.24it/s]


Extracting /root/.pytorch/MNIST_data/MNIST/raw/t10k-labels-idx1-ubyte.gz to /root/.pytorch/MNIST_data/MNIST/raw



In [6]:
# Create data loaders
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=100, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=100, shuffle=False)

In [7]:
# Perform PCA on training data
pca = PCA(n_components=dim_pca)
x_train_pca = pca.fit_transform(train_dataset.data.numpy().reshape(-1, 28*28))
x_test_pca = pca.transform(test_dataset.data.numpy().reshape(-1, 28*28))

In [8]:
# Scale PCA values to feasible range of local detuning
x_train_pca = x_train_pca / np.max(np.abs(x_train_pca)) * Δ_max
x_test_pca = x_test_pca / np.max(np.abs(x_test_pca)) * Δ_max

In [9]:
x_test_pca[:, 1:num_examples]

array([[ 1.88088648, -0.10776461, -0.7826744 , ...,  1.18408869,
        -1.00410033, -0.46812388],
       [-2.40351577, -0.38411446,  1.00276653, ...,  0.58194055,
         0.36999777, -0.22138175],
       [-1.08367012,  0.16644878,  0.6542185 , ..., -0.27662055,
         0.24835942, -0.34184245],
       ...,
       [ 1.50126602,  0.89318494, -1.04370732, ..., -0.88679635,
        -0.06195629, -2.01705899],
       [-0.27316301,  1.61689029, -0.63723699, ...,  0.41753497,
         0.67927445, -0.54204674],
       [-0.22766542,  1.7760525 ,  2.12734337, ..., -1.06866973,
         0.43045607,  0.13664465]])

In [10]:
# One-hot encode labels
encoder = OneHotEncoder(sparse_output=False)
y_train = encoder.fit_transform(train_dataset.targets.numpy().reshape(-1, 1))
y_test = encoder.transform(test_dataset.targets.numpy().reshape(-1, 1))

In [11]:
y_test[:, 1:num_examples]

array([[0., 0., 0., ..., 1., 0., 0.],
       [0., 1., 0., ..., 0., 0., 0.],
       [1., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

In [12]:
# Define quantum reservoir computing (QRC) layer
class DetuningLayer(nn.Module):
    def __init__(self, atoms, readouts, Ω, t_start, t_end, step):
        super(DetuningLayer, self).__init__()
        self.atoms = atoms
        self.readouts = readouts
        self.Ω = Ω
        self.t_start = t_start
        self.t_end = t_end
        self.step = step
    def apply_layer(self, x):
      run_time = var("run_time")

      @waveform(run_time + 0.2)
      def delta(t, amp, omega):
        return np.sin(omega * t) * amp
      # example ham
      delta = delta.sample(0.05, "linear")
      ampl = piecewise_linear([0.1, run_time, 0.1], [0, 10, 10, 0])
      phase = piecewise_constant([2, 2], [0, np.pi])
      register = start.add_position((0, 0))
      atoms_position = (0, 0)
      # note x is with predict no just x
      prog = rydberg_h(
          self.atoms, detuning=x, phase=self.Ω, batch_params={}
      )
    # Simulate quantum dynamics and compute readouts
    # have to use bloqade quantum
    # This part is not implemented in Python, as it requires a quantum simulator
    # calculating steps
      # the hamiltonian would be calculate through the rydberg_h function given in bloqade python
      # KrylovEvolution(reg, layer.t_start:layer.step:layer.t_end, h)
      # question this does not exist so this has to implemented, any specific suggestions regarding this? I did found one funded by unitary fund https://github.com/emilianomfortes/krylovsolver
      # for (step, reg, _) in prob ## store the expectation of each operator for the given state in the output vector
      # question: how to obtain state in polynomial time since state tomography takes exponential measurements losing Q advantage
      # push!(readouts, chain(put(nsites, i => Z), put(nsites, j => Z))) ## what does this mean? Something like CZ?
      self.atoms @ np.exp(-1j * h * (self.t_end - self.t_start))

In [ ]:
# # need to figure out how to access this stuff
# TaskData()
# BloqadeEmulation()

TypeError: BloqadeEmulation.__init__() missing 1 required positional argument: 'task_data'

In [18]:
# selecting a simple atom config, can also use variable config
atoms = Chain(2, lattice_spacing=10)

In [31]:
t_start = 0
t_end = 5
t_step = 0.1
steps = (t_end - t_start)/t_step + 1
times = []
for i in range(int(steps)):
  times.append(t_start + i*t_step)

In [29]:
# times

In [32]:

[emu] = (
    # start.add_position([(0, 0), (0, 5), (0, 10)])
    atoms # here we can use different lattice configurations
    .rydberg.rabi.amplitude.uniform.constant(15.0, 4.0) # here idk since omega in the julia code was frequency but here there is amplitude and phase
    .detuning.uniform.constant(1.0, 4.0) # here we can use x_train variable
    .bloqade
    .python() #bloqademulation
    .hamiltonian()
)
emu.evolve(times=times) # here evolution time list is used as input
# data = np.zeros(8)
# data[0] = 1

#### How they do zero state
# emu.zero_state()

<generator object AnalogGate._apply at 0x7dfbb78a7df0>

In [ ]:
# have a certain square lattice or something. Don't know how much the shape matters.
atoms = Chain(2, lattice_spacing="distance") # probably generate_sites(ChainLattice(), dim_pca; scale = d); # put atoms in a chain with 10 micron spacing
nsites = dim_pca
Ω = 2*np.pi
t_start = 0.0
t_end = 4.0
step = 0.5
#using space class we have to establish zero state for the register
reg = zero_state(nsites)
#readouts seems non-trivial
# layer = DetuningLayer()

NameError: name 'zero_state' is not defined

## Defining a NN model

In [ ]:
# Define neural network model
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.fc1 = nn.Linear(dim_pca, 10)
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        return x
    # Train classical model using PCA features
    model_reg = Net()
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model_reg.parameters(), lr=0.01)
    for epoch in range(1000):
        for x, y in train_loader:
            x = x.view(-1, 28*28)
            x_pca = pca.transform(x.numpy())
            x_pca = torch.tensor(x_pca, dtype=torch.float32)
            y = torch.tensor(y, dtype=torch.long)
            optimizer.zero_grad()
            output = model_reg(x_pca)
            loss = criterion(output, y)
            loss.backward()
            optimizer.step()
    # Train QRC model using quantum reservoir computing
    pre_layer = DetuningLayer(atoms, readouts, Ω, t_start, t_end, step)
    model_qrc = Net()
    for epoch in range(1000):
        for x, y in train_loader:
            x = x.view(-1, 28*28)
            x_pca = pca.transform(x.numpy())
